# Attention and the Transformer from Scratch (PyTorch)

Following [D2L Ch. 11 Attention Mechanisms and Transformers](https://d2l.ai/chapter_attention-mechanisms-and-transformers/index.html) and [Vaswani et al., 2017 — *Attention Is All You Need*](https://arxiv.org/abs/1706.03762).

We build every piece of the Transformer by hand and train it to translate **English → French**.

**Contents**
1. Scaled dot-product attention (with masking)
2. Multi-head attention
3. Positional encoding
4. Encoder and decoder blocks
5. Data: English–French sentence pairs
6. Training, greedy decoding and BLEU
7. Visualizing attention weights

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/lee2716/Dive_into_Deep_Learning_Code_Implementation/blob/main/transformer.ipynb)

In [ ]:
import collections
import math
import os
import time
import urllib.request
import zipfile

import torch
from torch import nn
from torch.nn import functional as F
import matplotlib.pyplot as plt

torch.manual_seed(0)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)

## 1. Scaled Dot-Product Attention

Each **query** is compared with every **key**; the similarities become weights (softmax) for averaging the **values**:

$$\mathrm{Attention}(\mathbf{Q}, \mathbf{K}, \mathbf{V}) = \mathrm{softmax}\left(\frac{\mathbf{Q}\mathbf{K}^\top}{\sqrt{d}}\right)\mathbf{V}$$

Dividing by $\sqrt{d}$ keeps the dot products from growing with the dimension $d$, which would saturate the softmax.

A boolean **mask** removes positions a query must not look at — padding tokens, or (in the decoder) future tokens. Masked scores are set to $-\infty$ before the softmax, so they get weight 0.

In [ ]:
def scaled_dot_product_attention(Q, K, V, mask=None):
    """Q: (..., n_q, d), K: (..., n_k, d), V: (..., n_k, d_v), mask: broadcastable to (..., n_q, n_k), True = keep."""
    d = Q.shape[-1]
    scores = Q @ K.transpose(-2, -1) / math.sqrt(d)
    if mask is not None:
        scores = scores.masked_fill(~mask, float('-inf'))
    weights = F.softmax(scores, dim=-1)
    return weights @ V, weights


# toy example: 1 query, 4 keys, the last 2 keys are padding
Q = torch.randn(1, 1, 8)
K = torch.randn(1, 4, 8)
V = torch.arange(4.0).reshape(1, 4, 1)
mask = torch.tensor([True, True, False, False]).reshape(1, 1, 4)
out, w = scaled_dot_product_attention(Q, K, V, mask)
print('weights:', w)       # last two are 0
print('output :', out)

## 2. Multi-Head Attention

Instead of one attention over $d_\text{model}$-dimensional vectors, project Q, K, V into $h$ smaller subspaces, attend in each **head** in parallel, then concatenate and project back. Different heads can learn different kinds of relationships (e.g. nearby words vs. subject–verb links).

Implementation trick: one linear layer per Q/K/V produces all heads at once; we reshape `(batch, seq, d_model)` → `(batch, heads, seq, d_model / heads)`.

In [ ]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads, dropout):
        super().__init__()
        assert d_model % num_heads == 0
        self.num_heads = num_heads
        self.W_q = nn.Linear(d_model, d_model, bias=False)
        self.W_k = nn.Linear(d_model, d_model, bias=False)
        self.W_v = nn.Linear(d_model, d_model, bias=False)
        self.W_o = nn.Linear(d_model, d_model, bias=False)
        self.dropout = nn.Dropout(dropout)
        self.attention_weights = None

    def split_heads(self, X):
        b, n, d = X.shape
        return X.reshape(b, n, self.num_heads, d // self.num_heads).transpose(1, 2)

    def forward(self, queries, keys, values, mask=None):
        Q = self.split_heads(self.W_q(queries))
        K = self.split_heads(self.W_k(keys))
        V = self.split_heads(self.W_v(values))
        if mask is not None:
            mask = mask.unsqueeze(1)                    # broadcast over heads
        out, self.attention_weights = scaled_dot_product_attention(Q, K, V, mask)
        out = out.transpose(1, 2).reshape(queries.shape)  # concatenate heads
        return self.W_o(self.dropout(out))


attn = MultiHeadAttention(d_model=32, num_heads=4, dropout=0.0)
X = torch.randn(2, 5, 32)
print(attn(X, X, X).shape, attn.attention_weights.shape)

## 3. Positional Encoding

Attention treats its input as a **set** — shuffling the tokens shuffles the output the same way. To give the model word order, we add a fixed sinusoidal signal to each position $i$:

$$p_{i, 2j} = \sin\left(\frac{i}{10000^{2j/d}}\right), \qquad p_{i, 2j+1} = \cos\left(\frac{i}{10000^{2j/d}}\right)$$

Low dimensions oscillate quickly, high dimensions slowly — like the bits of a binary counter, but continuous.

In [ ]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, dropout, max_len=1000):
        super().__init__()
        self.dropout = nn.Dropout(dropout)
        pos = torch.arange(max_len, dtype=torch.float32).reshape(-1, 1)
        div = torch.pow(10000, torch.arange(0, d_model, 2, dtype=torch.float32) / d_model)
        P = torch.zeros(1, max_len, d_model)
        P[0, :, 0::2] = torch.sin(pos / div)
        P[0, :, 1::2] = torch.cos(pos / div)
        self.register_buffer('P', P)

    def forward(self, X):
        return self.dropout(X + self.P[:, :X.shape[1]])


pe = PositionalEncoding(32, 0)
plt.figure(figsize=(8, 3))
for j in (6, 7, 8, 9):
    plt.plot(pe.P[0, :60, j], label=f'dim {j}')
plt.xlabel('position'); plt.legend(); plt.title('Positional encoding'); plt.show()

## 4. Encoder and Decoder Blocks

Both stacks are built from the same parts, each wrapped in a **residual connection + layer normalization** ("Add & Norm"):
- a **position-wise feed-forward network**: the same 2-layer MLP applied to every position;
- **self-attention** (encoder: each source token attends to all source tokens);
- in the decoder: **masked self-attention** (a target token may only look at earlier target tokens) and **encoder–decoder attention** (target tokens attend to the encoder output).

In [ ]:
class PositionWiseFFN(nn.Module):
    def __init__(self, d_model, d_ffn):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(d_model, d_ffn), nn.ReLU(), nn.Linear(d_ffn, d_model))

    def forward(self, X):
        return self.net(X)


class AddNorm(nn.Module):
    def __init__(self, d_model, dropout):
        super().__init__()
        self.dropout = nn.Dropout(dropout)
        self.ln = nn.LayerNorm(d_model)

    def forward(self, X, Y):
        return self.ln(X + self.dropout(Y))


class EncoderBlock(nn.Module):
    def __init__(self, d_model, d_ffn, num_heads, dropout):
        super().__init__()
        self.attention = MultiHeadAttention(d_model, num_heads, dropout)
        self.addnorm1 = AddNorm(d_model, dropout)
        self.ffn = PositionWiseFFN(d_model, d_ffn)
        self.addnorm2 = AddNorm(d_model, dropout)

    def forward(self, X, src_mask):
        Y = self.addnorm1(X, self.attention(X, X, X, src_mask))
        return self.addnorm2(Y, self.ffn(Y))


class DecoderBlock(nn.Module):
    def __init__(self, d_model, d_ffn, num_heads, dropout):
        super().__init__()
        self.self_attention = MultiHeadAttention(d_model, num_heads, dropout)
        self.addnorm1 = AddNorm(d_model, dropout)
        self.cross_attention = MultiHeadAttention(d_model, num_heads, dropout)
        self.addnorm2 = AddNorm(d_model, dropout)
        self.ffn = PositionWiseFFN(d_model, d_ffn)
        self.addnorm3 = AddNorm(d_model, dropout)

    def forward(self, X, enc_out, tgt_mask, src_mask):
        Y = self.addnorm1(X, self.self_attention(X, X, X, tgt_mask))
        Z = self.addnorm2(Y, self.cross_attention(Y, enc_out, enc_out, src_mask))
        return self.addnorm3(Z, self.ffn(Z))

### The full model

- **Source mask** `(batch, 1, src_len)`: hide padding tokens of the source.
- **Target mask** `(batch, tgt_len, tgt_len)`: hide padding *and* future positions (lower-triangular "causal" mask).
- Embeddings are multiplied by $\sqrt{d_\text{model}}$ so they have a similar scale to the positional encoding.

In [ ]:
class Transformer(nn.Module):
    def __init__(self, src_vocab, tgt_vocab, d_model=256, d_ffn=512, num_heads=4,
                 num_layers=2, dropout=0.2, pad=0):
        super().__init__()
        self.pad, self.d_model = pad, d_model
        self.src_emb = nn.Embedding(src_vocab, d_model)
        self.tgt_emb = nn.Embedding(tgt_vocab, d_model)
        self.pos = PositionalEncoding(d_model, dropout)
        self.encoder = nn.ModuleList([EncoderBlock(d_model, d_ffn, num_heads, dropout) for _ in range(num_layers)])
        self.decoder = nn.ModuleList([DecoderBlock(d_model, d_ffn, num_heads, dropout) for _ in range(num_layers)])
        self.out = nn.Linear(d_model, tgt_vocab)

    def src_mask(self, src):
        return (src != self.pad).unsqueeze(1)                              # (b, 1, n_src)

    def tgt_mask(self, tgt):
        n = tgt.shape[1]
        causal = torch.tril(torch.ones(n, n, dtype=torch.bool, device=tgt.device))
        return (tgt != self.pad).unsqueeze(1) & causal                      # (b, n_tgt, n_tgt)

    def encode(self, src):
        mask = self.src_mask(src)
        X = self.pos(self.src_emb(src) * math.sqrt(self.d_model))
        for blk in self.encoder:
            X = blk(X, mask)
        return X, mask

    def decode(self, tgt, enc_out, src_mask):
        X = self.pos(self.tgt_emb(tgt) * math.sqrt(self.d_model))
        tgt_mask = self.tgt_mask(tgt)
        for blk in self.decoder:
            X = blk(X, enc_out, tgt_mask, src_mask)
        return self.out(X)

    def forward(self, src, tgt_in):
        enc_out, src_mask = self.encode(src)
        return self.decode(tgt_in, enc_out, src_mask)

## 5. Data: English–French Sentence Pairs

The [Tatoeba](https://tatoeba.org/) pairs packaged by D2L. Preprocessing: lowercase, put spaces around punctuation, split into words. Words seen fewer than twice become `<unk>`. Every sentence gets an `<eos>` token and is truncated / padded to `num_steps` tokens.

The file is sorted from short to long sentences; we use the first 20,000 pairs (short sentences) so training is quick even on a CPU.

In [ ]:
URL = 'http://d2l-data.s3-accelerate.amazonaws.com/fra-eng.zip'
os.makedirs('data', exist_ok=True)
zip_path = os.path.join('data', 'fra-eng.zip')
if not os.path.exists(zip_path):
    urllib.request.urlretrieve(URL, zip_path)
with zipfile.ZipFile(zip_path) as z:
    raw = z.read('fra-eng/fra.txt').decode('utf-8')
print(raw[:120])


def preprocess(text):
    text = text.replace(' ', ' ').replace('\xa0', ' ').lower()
    out = []
    for i, ch in enumerate(text):
        if ch in ',.!?' and i > 0 and text[i - 1] != ' ':
            out.append(' ')
        out.append(ch)
    return ''.join(out)


def tokenize(text, num_examples):
    src, tgt = [], []
    for line in text.split('\n')[:num_examples]:
        parts = line.split('\t')
        if len(parts) == 2:
            src.append(parts[0].split())
            tgt.append(parts[1].split())
    return src, tgt


src_sents, tgt_sents = tokenize(preprocess(raw), num_examples=20000)
mid = len(src_sents) // 2
print(src_sents[mid], tgt_sents[mid])

In [ ]:
class Vocab:
    def __init__(self, sentences, min_freq=2, reserved=('<pad>', '<bos>', '<eos>', '<unk>')):
        counter = collections.Counter(t for s in sentences for t in s)
        self.idx_to_token = list(reserved) + sorted(t for t, c in counter.items() if c >= min_freq)
        self.token_to_idx = {t: i for i, t in enumerate(self.idx_to_token)}
        self.unk = self.token_to_idx['<unk>']

    def __len__(self):
        return len(self.idx_to_token)

    def __getitem__(self, tokens):
        return [self.token_to_idx.get(t, self.unk) for t in tokens]

    def to_tokens(self, indices):
        return [self.idx_to_token[int(i)] for i in indices]


src_vocab, tgt_vocab = Vocab(src_sents), Vocab(tgt_sents)
PAD, BOS, EOS = 0, 1, 2
num_steps = 12


def to_tensor(sentences, vocab):
    rows = []
    for s in sentences:
        ids = vocab[s][:num_steps - 1] + [EOS]
        rows.append(ids + [PAD] * (num_steps - len(ids)))
    return torch.tensor(rows)


src_data, tgt_data = to_tensor(src_sents, src_vocab), to_tensor(tgt_sents, tgt_vocab)
print('source vocab:', len(src_vocab), '| target vocab:', len(tgt_vocab), '| data:', tuple(src_data.shape))

perm = torch.randperm(len(src_data))
val_idx, train_idx = perm[:1000], perm[1000:]
train_loader = torch.utils.data.DataLoader(
    torch.utils.data.TensorDataset(src_data[train_idx], tgt_data[train_idx]), batch_size=128, shuffle=True)

## 6. Training

**Teacher forcing:** the decoder input is the target shifted right by one (`<bos>` + target without its last token), and the model learns to predict the next token at every position at once — the causal mask stops it from cheating.

Padding positions are excluded from the loss (`ignore_index=PAD`). We use Adam with gradient clipping and **label smoothing** (as in the original paper).

In [ ]:
model = Transformer(len(src_vocab), len(tgt_vocab)).to(device)
print(f'{sum(p.numel() for p in model.parameters()):,} parameters')
optimizer = torch.optim.Adam(model.parameters(), lr=5e-4)
loss_fn = nn.CrossEntropyLoss(ignore_index=PAD, label_smoothing=0.1)


def shift_right(tgt):
    bos = torch.full((tgt.shape[0], 1), BOS, device=tgt.device)
    return torch.cat([bos, tgt[:, :-1]], dim=1)


num_epochs, losses = 20, []
start = time.time()
for epoch in range(num_epochs):
    model.train()
    total, n = 0.0, 0
    for src, tgt in train_loader:
        src, tgt = src.to(device), tgt.to(device)
        logits = model(src, shift_right(tgt))
        loss = loss_fn(logits.reshape(-1, logits.shape[-1]), tgt.reshape(-1))
        optimizer.zero_grad()
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        tokens = (tgt != PAD).sum().item()
        total += loss.item() * tokens
        n += tokens
    losses.append(total / n)
    if (epoch + 1) % 2 == 0:
        print(f'epoch {epoch + 1:2d}: loss {losses[-1]:.3f}  ({time.time() - start:.0f}s)')

plt.plot(range(1, num_epochs + 1), losses)
plt.xlabel('epoch'); plt.ylabel('loss'); plt.grid(True); plt.show()

### Greedy decoding

At prediction time we don't know the target, so we generate it one token at a time: start from `<bos>`, run the decoder, append the most likely next token, and stop at `<eos>`. (Re-running the decoder on the whole prefix each step is simple; production code caches the keys/values of earlier steps.)

In [ ]:
def translate(sentence, model):
    model.eval()
    tokens = preprocess(sentence).split()
    src = to_tensor([tokens], src_vocab).to(device)
    with torch.no_grad():
        enc_out, src_mask = model.encode(src)
        out = torch.tensor([[BOS]], device=device)
        for _ in range(num_steps):
            logits = model.decode(out, enc_out, src_mask)
            next_tok = logits[0, -1].argmax().item()
            out = torch.cat([out, torch.tensor([[next_tok]], device=device)], dim=1)
            if next_tok == EOS:
                break
    result = [t for t in tgt_vocab.to_tokens(out[0, 1:]) if t != '<eos>']
    return ' '.join(result)

### BLEU

BLEU compares n-gram overlap between prediction and reference, with a penalty for translations that are too short. 1.0 means a perfect match.

$$\mathrm{BLEU} = \exp\left(\min\left(0, 1 - \frac{\mathrm{len}_\text{label}}{\mathrm{len}_\text{pred}}\right)\right) \prod_{n=1}^{k} p_n^{1/2^n}$$

In [ ]:
def bleu(pred, label, k=2):
    pred_tokens, label_tokens = pred.split(), label.split()
    len_pred, len_label = len(pred_tokens), len(label_tokens)
    if len_pred == 0:
        return 0.0
    score = math.exp(min(0, 1 - len_label / len_pred))
    for n in range(1, min(k, len_pred) + 1):
        num_matches, label_subs = 0, collections.Counter(
            ' '.join(label_tokens[i: i + n]) for i in range(len_label - n + 1))
        for i in range(len_pred - n + 1):
            ngram = ' '.join(pred_tokens[i: i + n])
            if label_subs[ngram] > 0:
                num_matches += 1
                label_subs[ngram] -= 1
        score *= math.pow(num_matches / (len_pred - n + 1), math.pow(0.5, n))
    return score


examples = [('go .', 'va !'), ('i lost .', "j'ai perdu ."), ("he's calm .", 'il est calme .'),
            ("i'm home .", 'je suis chez moi .')]
for en, fr in examples:
    pred = translate(en, model)
    print(f'{en:12s} => {pred:25s} BLEU {bleu(pred, fr):.3f}')

Average BLEU on the 1,000 held-out pairs the model never trained on:

In [ ]:
scores = []
for i in val_idx.tolist():
    pred = translate(' '.join(src_sents[i]), model)
    scores.append(bleu(pred, ' '.join(tgt_sents[i])))
print(f'validation BLEU-2: {sum(scores) / len(scores):.3f}')

for i in val_idx[:8].tolist():
    en = ' '.join(src_sents[i])
    print(f'{en:28s} => {translate(en, model):30s} (ref: {" ".join(tgt_sents[i])})')

## 7. Visualizing Attention

The encoder–decoder attention weights of the last decoder layer show which source word each generated French word looks at. (Averaged over heads.)

In [ ]:
sentence = "i'm home ."
translation = translate(sentence, model)
src = to_tensor([preprocess(sentence).split()], src_vocab).to(device)
tgt_in = torch.tensor([[BOS] + tgt_vocab[translation.split()]], device=device)
with torch.no_grad():
    model(src, tgt_in)
weights = model.decoder[-1].cross_attention.attention_weights[0].mean(dim=0).cpu()

src_tokens = preprocess(sentence).split() + ['<eos>']
tgt_tokens = ['<bos>'] + translation.split()
plt.imshow(weights[:, :len(src_tokens)], cmap='Reds')
plt.xticks(range(len(src_tokens)), src_tokens)
plt.yticks(range(len(tgt_tokens)), tgt_tokens)
plt.xlabel('source (English)'); plt.ylabel('decoder input (French)')
plt.colorbar(); plt.title('Encoder–decoder attention'); plt.show()

## Exercises

1. Train with `num_layers=1` and `num_heads=1`. How do loss and BLEU change?
2. Remove the positional encoding. Can the model still translate? Why does word order matter less for very short sentences?
3. Replace our blocks with `nn.Transformer` from PyTorch and check you get similar results.
4. Implement beam search and compare it with greedy decoding.